In [1]:

# AUTONOMOUS AUCTION BIDDING ECOSYSTEM — PHASE 2 THROUGH PHASE 7
#   phase 1 = completed i.e model price prediction
#   Phase 2 =  Elite Strategic Bidding Engine
#   Phase 3 = Memory-Driven Adaptive Learning
#   Phase 4 = Advanced Strategic Reasoning
#   Phase 5 = True Multi-Agent Ecosystem
#   Phase 6 = Future Market Intelligence (LSTM)
#   Phase 7 = Meta-Strategic Governance

import os
import pickle
import warnings
import json
import time
import math
import logging
from collections import deque, defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any
from enum import Enum

import numpy as np
import pandas as pd



warnings.filterwarnings("ignore")

# pytorch for lstm
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

# caatbosst 
try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s"
)
logger = logging.getLogger("AuctionEcosystem")

# Shared terms and enumeration used throughout 

class MarketRegime(Enum):
    """Current perceived market condition."""
    COLD      = "cold"       # low competition, low prices
    NORMAL    = "normal"     # baseline conditions
    WARM      = "warm"       # moderate competition
    HOT       = "hot"        # aggressive competition
    OVERHEATED= "overheated" # prices exceed intrinsic value

class AuctionStage(Enum):
    """Where we are in the session."""
    OPENING    = "opening"    # first 20% of session
    MID        = "mid"        # 20–70%
    LATE       = "late"       # 70–90%
    CLOSING    = "closing"    # final 10%

class VehicleSegment(Enum):
    """Vehicle market segment."""
    ECONOMY    = "economy"
    MID_RANGE  = "mid_range"
    LUXURY     = "luxury"
    EXOTIC     = "exotic"

LUXURY_BRANDS = {
    "bmw", "audi", "mercedes-benz", "lexus", "porsche",
    "jaguar", "land rover", "cadillac", "lincoln", "infiniti",
    "acura", "volvo", "genesis", "alfa romeo"
}
EXOTIC_BRANDS = {"ferrari", "lamborghini", "maserati", "bentley", "rolls-royce", "aston martin"}
PREFERRED_COLORS = {"black", "white", "silver", "gray", "grey"}
PREFERRED_INTERIORS = {"black", "gray", "grey", "beige", "tan"}

# PHASE 3 — MEMORY-DRIVEN ADAPTIVE LEARNING
# Key ideas: reward learning, failure memory, strategy evolution,
#             behavioral adaptation, memory retrieval
# Model stores every action result , learn from it , extract apttern and adopt acc behavior 

@dataclass
class AuctionRecord:

    car_id:             str
    features:           Dict[str, Any]
    predicted_value:    float
    actual_price:       float
    our_final_bid:      float
    won:                bool
    profit_if_won:      float
    margin_pct:         float
    market_regime:      str
    auction_stage:      str
    vehicle_segment:    str
    rounds_taken:       int
    competition_proxy:  float      # how quickly price escalated
    timestamp:          float = field(default_factory=time.time)
    regret:             float = 0.0  # opportunity cost if lost

    def to_feature_vector(self) -> np.ndarray:
        """Convert to numeric vector for similarity search."""
        return np.array([
            self.predicted_value / 50000,
            self.margin_pct,
            self.competition_proxy,
            1.0 if self.won else 0.0,
            self.profit_if_won / 5000 if self.won else 0.0,
            self.rounds_taken / 15.0,
        ])


class EpisodicMemory:


    def __init__(self, max_episodes: int = 500):
        self.episodes: deque = deque(maxlen=max_episodes)
        self.wins:  List[AuctionRecord] = []
        self.losses: List[AuctionRecord] = []

    def store(self, record: AuctionRecord):
        self.episodes.append(record)
        if record.won:
            self.wins.append(record)
        else:
            self.losses.append(record)

    def retrieve_similar(
        self,
        query_value: float,
        query_segment: str,
        query_regime: str,
        top_k: int = 5
    ) -> List[AuctionRecord]:

        if not self.episodes:
            return []

        candidates = [
            ep for ep in self.episodes
            if ep.vehicle_segment == query_segment
        ]
        if not candidates:
            candidates = list(self.episodes)

        # Score by price proximity (normalized)
        scored = []
        for ep in candidates:
            price_sim = 1.0 / (1.0 + abs(ep.predicted_value - query_value) / max(query_value, 1))
            regime_bonus = 0.3 if ep.market_regime == query_regime else 0.0
            scored.append((price_sim + regime_bonus, ep))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [ep for _, ep in scored[:top_k]]

    def win_rate(self, segment: Optional[str] = None) -> float:
        eps = [e for e in self.episodes if e.vehicle_segment == segment] if segment else list(self.episodes)
        if not eps: return 0.5
        return sum(1 for e in eps if e.won) / len(eps)

    def avg_margin_on_wins(self, segment: Optional[str] = None) -> float:
        wins = [e for e in self.wins if e.vehicle_segment == segment] if segment else self.wins
        if not wins: return 0.10
        return np.mean([e.margin_pct for e in wins])

    def avg_competition_proxy(self) -> float:
        if not self.episodes: return 0.5
        return np.mean([e.competition_proxy for e in list(self.episodes)[-20:]])

    def failure_analysis(self) -> Dict[str, float]:

        if not self.losses:
            return {}
        recent_losses = list(self.losses)[-30:]
        analysis = {
            "avg_loss_margin":      np.mean([abs(e.margin_pct) for e in recent_losses]),
            "hot_market_loss_rate": sum(1 for e in recent_losses if e.market_regime in ("hot","overheated")) / len(recent_losses),
            "luxury_loss_rate":     sum(1 for e in recent_losses if e.vehicle_segment == "luxury") / len(recent_losses),
            "late_stage_loss_rate": sum(1 for e in recent_losses if e.auction_stage == "closing") / len(recent_losses),
        }
        return analysis


class AdaptivePolicyEngine:


    def __init__(self):
        # Mutable policy state
        self.base_margin       = 0.10
        self.aggression_scale  = 1.0
        self.skip_threshold    = 0.88
        self.luxury_tolerance  = 0.05
        self.strategy_mode     = "balanced"   # "conservative" | "balanced" | "aggressive"

        # Learning rate
        self._lr               = 0.05

        # Regime-specific adjustments (learned over time)
        self._regime_adjustments: Dict[str, float] = defaultdict(float)

    def update(self, record: AuctionRecord, memory: EpisodicMemory):
        """
        Update policy parameters based on the most recent auction outcome.
        Uses a reward signal + failure pattern analysis.
        """
        reward = self._compute_reward(record)

        # Margin adaptation 
        if not record.won and record.our_final_bid < record.actual_price:
            # We lost — could have won by bidding more
            opportunity_cost = (record.profit_if_won * 0.5) if record.profit_if_won > 0 else 0
            if opportunity_cost > 0:
                # Slightly lower margin requirement (be more willing to bid)
                self.base_margin = max(0.05, self.base_margin - self._lr * 0.3)
        elif record.won and record.margin_pct < self.base_margin * 0.5:
            # Won but thin margin — increase required margin
            self.base_margin = min(0.25, self.base_margin + self._lr * 0.5)

        # Aggression adaptation 
        failure_info = memory.failure_analysis()
        if failure_info.get("hot_market_loss_rate", 0) > 0.6:
            # Hot markets causing losses — reduce aggression
            self.aggression_scale = max(0.6, self.aggression_scale - self._lr * 0.2)
        elif memory.win_rate() > 0.65 and reward > 0:
            # High win rate + positive reward — can afford to be more selective
            self.aggression_scale = min(1.5, self.aggression_scale + self._lr * 0.1)

        # Strategy mode 
        recent_win_rate = memory.win_rate()
        bankroll_ratio  = getattr(record, "_bankroll_ratio", 1.0)
        if bankroll_ratio < 0.40 or recent_win_rate < 0.30:
            self.strategy_mode = "conservative"
        elif recent_win_rate > 0.60 and bankroll_ratio > 0.70:
            self.strategy_mode = "aggressive"
        else:
            self.strategy_mode = "balanced"

        #  Regime-specific learning 
        regime_key = record.market_regime
        regime_reward = reward if record.won else -abs(reward) * 0.5
        self._regime_adjustments[regime_key] = (
            0.8 * self._regime_adjustments[regime_key] + 0.2 * regime_reward
        )

        logger.debug(
            f"Policy updated: margin={self.base_margin:.3f} "
            f"aggression={self.aggression_scale:.3f} "
            f"mode={self.strategy_mode}"
        )

    def regime_margin_adjustment(self, regime: str) -> float:
        """Return learned margin adjustment for a specific market regime."""
        return self._regime_adjustments.get(regime, 0.0) * 0.02

    def _compute_reward(self, record: AuctionRecord) -> float:
        """
        Reward signal: positive for profitable wins, negative for losses
        that had positive profit potential.
        """
        if record.won:
            return record.profit_if_won / max(record.predicted_value, 1)
        else:
            return -record.regret / max(record.predicted_value, 1)

# PHASE 6 — FUTURE MARKET INTELLIGENCE
# future market was predicted using time series n analysis concept i.e lstm..
# Key ideas: multivariate LSTM, regime forecasting, opportunity density,
#             volatility estimation, future liquidity


class MarketLSTM(nn.Module if TORCH_AVAILABLE else object):


    def __init__(self, input_size: int = 6, hidden_size: int = 64,
                 num_layers: int = 2, output_size: int = 4):
        if not TORCH_AVAILABLE:
            return
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(0.3)
        self.fc      = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x)
        out = self.dropout(lstm_out[:, -1, :])   # last timestep
        return self.fc(out)


class FutureMarketIntelligence:

   # Phase 6: Forecasts future market conditions to enable
   # opportunity-preserving, temporally-aware bidding decisions.



    SEQ_LEN = 10  # lookback window

    def __init__(self):
        self._market_history: deque = deque(maxlen=200)
        self._model: Optional[Any] = None
        self._model_trained = False
        self._training_buffer: List[np.ndarray] = []

        # Fallback statistical forecasts (used when LSTM not trained)
        self._price_ema     = None    # exponential moving average of prices
        self._vol_ema       = None    # EMA of price volatility
        self._comp_ema      = None    # EMA of competition intensity

        # Forecast output (cached, refreshed each auction)
        self.forecast: Dict[str, float] = {
            "future_regime_hot_prob":       0.25,
            "opportunity_density":          0.5,
            "price_trend":                  0.0,
            "volatility":                   0.15,
            "future_opportunity_prob":      0.4,
            "expected_future_profit_score": 0.5,
        }

    def update(self, auction_price: float, competition_intensity: float,
               vehicle_quality: float, won: bool, regime: str):

        obs = np.array([
            auction_price / 50000,         # normalized price
            competition_intensity,
            vehicle_quality,
            1.0 if won else 0.0,
            {"cold":0,"normal":0.33,"warm":0.66,"hot":1,"overheated":1}
                .get(regime, 0.33),
            len(self._market_history) / 200  # session progress
        ], dtype=np.float32)

        self._market_history.append(obs)

        # Update EMAs
        alpha = 0.15
        if self._price_ema is None:
            self._price_ema = auction_price
            self._vol_ema   = 0.1
            self._comp_ema  = competition_intensity
        else:
            self._price_ema = (1 - alpha) * self._price_ema + alpha * auction_price
            self._vol_ema   = (1 - alpha) * self._vol_ema   + alpha * abs(auction_price - self._price_ema)
            self._comp_ema  = (1 - alpha) * self._comp_ema  + alpha * competition_intensity

        # Refresh forecasts
        self._refresh_forecast()

        # Attempt LSTM training if enough data
        if len(self._market_history) >= self.SEQ_LEN + 5 and TORCH_AVAILABLE:
            self._try_train_lstm()

    def _refresh_forecast(self):

        if self._model_trained and TORCH_AVAILABLE and len(self._market_history) >= self.SEQ_LEN:
            self._lstm_forecast()
        else:
            self._statistical_forecast()

    def _statistical_forecast(self):
        """EMA-based fallback forecast — always available."""
        if self._price_ema is None:
            return

        history = list(self._market_history)
        if len(history) < 3:
            return

        # Price trend (last 5 vs previous 5)
        recent    = np.mean([h[0] for h in history[-5:]])  if len(history) >= 5 else history[-1][0]
        prior     = np.mean([h[0] for h in history[-10:-5]]) if len(history) >= 10 else recent
        trend     = (recent - prior) / max(prior, 0.01)

        # Competition trend
        comp_recent = np.mean([h[1] for h in history[-5:]]) if len(history) >= 5 else 0.5

        # Hot regime probability (heuristic)
        hot_prob  = min(1.0, max(0.0, comp_recent * 0.6 + (1 if trend > 0.05 else 0) * 0.4))

        # Future opportunity: inverse of current market heat
        # (when market is hot, fewer bargains remain)
        future_opp = max(0.1, 1.0 - hot_prob * 0.7)

        vol = self._vol_ema / max(self._price_ema, 1) if self._price_ema else 0.15

        self.forecast.update({
            "future_regime_hot_prob":       hot_prob,
            "opportunity_density":          1.0 - comp_recent,
            "price_trend":                  trend,
            "volatility":                   min(1.0, vol),
            "future_opportunity_prob":      future_opp,
            "expected_future_profit_score": future_opp * (1.0 - vol),
        })

    def _lstm_forecast(self):
        """LSTM-based forecast when model is trained."""
        if not TORCH_AVAILABLE or not self._model_trained:
            return
        try:
            hist = list(self._market_history)[-self.SEQ_LEN:]
            seq  = torch.tensor([hist], dtype=torch.float32)
            with torch.no_grad():
                output = self._model(seq).squeeze().numpy()
            # output: [hot_prob, trend, volatility, opportunity_density]
            self.forecast.update({
                "future_regime_hot_prob":       float(np.clip(output[0], 0, 1)),
                "price_trend":                  float(np.clip(output[1], -0.5, 0.5)),
                "volatility":                   float(np.clip(output[2], 0, 1)),
                "opportunity_density":          float(np.clip(output[3], 0, 1)),
                "future_opportunity_prob":      float(np.clip(1 - output[0], 0, 1)),
                "expected_future_profit_score": float(np.clip(output[3] * (1 - output[2]), 0, 1)),
            })
        except Exception as e:
            logger.debug(f"LSTM forecast failed, using statistical: {e}")
            self._statistical_forecast()

    def _try_train_lstm(self):

        if not TORCH_AVAILABLE or len(self._market_history) < self.SEQ_LEN + 5:
            return
        try:
            history = np.array(list(self._market_history))
            X, y    = [], []
            for i in range(len(history) - self.SEQ_LEN - 1):
                X.append(history[i : i + self.SEQ_LEN])
                # Target: next hot_prob, price_trend, volatility, opp_density
                nxt       = history[i + self.SEQ_LEN]
                hot_proxy = nxt[1]              # competition_intensity as hot proxy
                trend_proxy = nxt[0] - history[i + self.SEQ_LEN - 1][0]  # price delta
                vol_proxy  = abs(trend_proxy)
                opp_proxy  = 1.0 - hot_proxy
                y.append([hot_proxy, trend_proxy, vol_proxy, opp_proxy])

            if len(X) < 5:
                return

            X_t = torch.tensor(X, dtype=torch.float32)
            y_t = torch.tensor(y, dtype=torch.float32)

            model     = MarketLSTM(input_size=6, hidden_size=32, num_layers=2, output_size=4)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
            criterion = nn.MSELoss()

            model.train()
            for _ in range(20):   # quick online training
                optimizer.zero_grad()
                pred = model(X_t)
                loss = criterion(pred, y_t)
                loss.backward()
                optimizer.step()

            self._model         = model
            self._model_trained = True
            logger.info("LSTM market intelligence model updated.")
        except Exception as e:
            logger.debug(f"LSTM training failed: {e}")

    def opportunity_cost_score(self, current_profit: float,
                                current_confidence: float) -> float:
        """
        Compute the opportunity cost of winning THIS auction vs
        preserving capital for future opportunities.

        opportunity_cost = E[future_profit] / current_profit

        If opportunity_cost > 1.0, future is more promising.
        If opportunity_cost < 0.5, take the current opportunity.

        Justification: This solves the "few trades, elite profit" mandate —
        the system correctly passes on mediocre trades when superior
        opportunities are likely forthcoming.
        """
        expected_future = self.forecast["expected_future_profit_score"]
        future_opp_prob = self.forecast["future_opportunity_prob"]

        # Expected future profit = base market intelligence * confidence in forecast
        forecast_confidence = min(1.0, len(self._market_history) / 50)
        expected_future_value = expected_future * future_opp_prob * forecast_confidence

        if current_profit <= 0:
            return 2.0  # no profit now → strongly prefer future

        return expected_future_value / (current_profit / 50000)


# PHASE 4 — ADVANCED STRATEGIC REASONING ENGINE
# deep reasoning for everything happening .
# Key ideas: sequential auction intelligence, opponent prediction,
#             future utility, dynamic liquidity, auction-stage awareness

@dataclass
class AuctionContext:

    #Complete state of the current auction decision context.
    #Passed through the reasoning pipeline as shared state.

    # Vehicle features
    car_features:           Dict[str, Any]
    predicted_value:        float
    vehicle_segment:        VehicleSegment

    # Bidding state
    bankroll:               float
    starting_bankroll:      float
    current_highest_bid:    float
    round_number:           int

    # Market context
    market_regime:          MarketRegime
    auction_stage:          AuctionStage
    session_position:       float    # 0.0–1.0, how far into session

    # Intelligence signals
    memory_records:         List[AuctionRecord]
    policy:                 "AdaptivePolicyEngine"
    forecast:               Dict[str, float]

    # Output: populated by reasoning nodes
    conviction_score:       float = 0.0
    max_bid:                float = 0.0
    skip_decision:          bool  = False
    skip_reason:            str   = ""
    bid_increment_override: Optional[float] = None
    reasoning_trace:        List[str] = field(default_factory=list)


class StrategicReasoningPipeline:


    def __init__(self, memory: EpisodicMemory, forecast_engine: FutureMarketIntelligence):
        self.memory   = memory
        self.forecast = forecast_engine

    # Node 1: Market Analysis 
    def _node_market_analysis(self, ctx: AuctionContext) -> AuctionContext:
        """
        Assess current market conditions.
        Outputs: market_heat_factor, regime_risk
        """
        regime = ctx.market_regime

        regime_risk_map = {
            MarketRegime.COLD:       0.10,
            MarketRegime.NORMAL:     0.25,
            MarketRegime.WARM:       0.40,
            MarketRegime.HOT:        0.65,
            MarketRegime.OVERHEATED: 0.85,
        }
        ctx._market_heat_factor = regime_risk_map.get(regime, 0.25)

        # Stage-based strategy adjustment
        stage_risk = {
            AuctionStage.OPENING: 0.2,
            AuctionStage.MID:     0.5,
            AuctionStage.LATE:    0.7,
            AuctionStage.CLOSING: 0.9,
        }
        ctx._stage_urgency = stage_risk.get(ctx.auction_stage, 0.5)

        ctx.reasoning_trace.append(
            f"[MarketAnalysis] Regime={regime.value} "
            f"heat={ctx._market_heat_factor:.2f} "
            f"stage_urgency={ctx._stage_urgency:.2f}"
        )
        return ctx

    #  Node 2: Risk Assessment
    def _node_risk_assessment(self, ctx: AuctionContext) -> AuctionContext:

        #Assess risk of this specific bid decision.
        #Considers: bankroll health, recent failure patterns, prediction confidence.

        bankroll_ratio = ctx.bankroll / max(ctx.starting_bankroll, 1)

        # Prediction confidence: higher if similar historical cases exist
        similar_cases = self.memory.retrieve_similar(
            ctx.predicted_value, ctx.vehicle_segment.value, ctx.market_regime.value
        )
        prediction_confidence = min(0.95, 0.4 + len(similar_cases) * 0.1)

        # Bankroll risk: exponentially increases as bankroll depletes
        bankroll_risk = math.exp(-3 * bankroll_ratio)

        # Failure pattern risk
        failure_info = self.memory.failure_analysis()
        segment_loss_risk = failure_info.get("luxury_loss_rate", 0.0) \
            if ctx.vehicle_segment == VehicleSegment.LUXURY else 0.0
        hot_market_risk = failure_info.get("hot_market_loss_rate", 0.0) \
            if ctx.market_regime in (MarketRegime.HOT, MarketRegime.OVERHEATED) else 0.0

        ctx._composite_risk = (
            0.40 * bankroll_risk +
            0.25 * ctx._market_heat_factor +
            0.20 * hot_market_risk +
            0.15 * segment_loss_risk
        )
        ctx._prediction_confidence = prediction_confidence
        ctx._bankroll_ratio        = bankroll_ratio

        # Hard skip: bankroll too low
        if bankroll_ratio < 0.20:
            ctx.skip_decision = True
            ctx.skip_reason   = f"Bankroll critically low ({bankroll_ratio:.1%})"

        ctx.reasoning_trace.append(
            f"[RiskAssessment] risk={ctx._composite_risk:.3f} "
            f"pred_confidence={prediction_confidence:.3f} "
            f"bankroll_ratio={bankroll_ratio:.2f}"
        )
        return ctx

    # Node 3: Opportunity Cost Analysis
    def _node_opportunity_cost(self, ctx: AuctionContext) -> AuctionContext:

        #QUES :- If to spend capital now or preserve for future.
        #related to  'few trades, elite profit' mandate.

        if ctx.skip_decision:
            return ctx

        potential_profit = ctx.predicted_value - ctx.current_highest_bid
        if potential_profit <= 0:
            ctx.skip_decision = True
            ctx.skip_reason   = "No profit potential at current bid level"
            return ctx

        opp_cost_ratio = self.forecast.opportunity_cost_score(
            current_profit=potential_profit,
            current_confidence=ctx._prediction_confidence
        )

        ctx._opportunity_cost_ratio = opp_cost_ratio

        # If future opportunities are significantly more promising AND
        # we're not in the closing stage, prefer to wait
        if opp_cost_ratio > 1.8 and ctx.auction_stage not in (AuctionStage.CLOSING,):
            if ctx.vehicle_segment != VehicleSegment.LUXURY:
                ctx.skip_decision = True
                ctx.skip_reason   = (
                    f"Future opportunity score ({opp_cost_ratio:.2f}x) "
                    f"exceeds current opportunity — preserving capital"
                )

        ctx.reasoning_trace.append(
            f"[OpportunityCost] ratio={opp_cost_ratio:.2f} "
            f"potential_profit=${potential_profit:,.0f} "
            f"skip={ctx.skip_decision}"
        )
        return ctx

    # Node 4: Conviction Scoring 
    def _node_conviction_scoring(self, ctx: AuctionContext) -> AuctionContext:

        if ctx.skip_decision:
            return ctx

        # Base conviction: how much value we expect to capture
        value_margin = (ctx.predicted_value - ctx.current_highest_bid) / max(ctx.predicted_value, 1)

        # Historical success signal
        seg_win_rate = self.memory.win_rate(ctx.vehicle_segment.value)

        # Memory signal: do similar historical cases predict win?
        similar = self.memory.retrieve_similar(
            ctx.predicted_value, ctx.vehicle_segment.value, ctx.market_regime.value
        )
        historical_win_rate_similar = (
            sum(1 for r in similar if r.won) / len(similar) if similar else 0.5
        )

        # Stage signal: late stage = fewer future bids, raise conviction
        stage_multiplier = {
            AuctionStage.OPENING: 0.8,
            AuctionStage.MID:     1.0,
            AuctionStage.LATE:    1.2,
            AuctionStage.CLOSING: 1.4,
        }.get(ctx.auction_stage, 1.0)

        # Luxury premium: luxury cars have higher resale certainty
        luxury_bonus = 0.15 if ctx.vehicle_segment == VehicleSegment.LUXURY else 0.0

        conviction = (
            0.35 * min(1.0, value_margin * 3) +
            0.20 * ctx._prediction_confidence +
            0.20 * historical_win_rate_similar +
            0.15 * seg_win_rate +
            0.10 * (1 - ctx._composite_risk) +
            luxury_bonus
        ) * stage_multiplier

        ctx.conviction_score = min(1.0, max(0.0, conviction))

        # Skip if conviction too low
        if ctx.conviction_score < 0.30:
            ctx.skip_decision = True
            ctx.skip_reason   = f"Conviction too low ({ctx.conviction_score:.2f} < 0.30)"

        ctx.reasoning_trace.append(
            f"[Conviction] score={ctx.conviction_score:.3f} "
            f"value_margin={value_margin:.3f} "
            f"hist_win_rate={historical_win_rate_similar:.3f}"
        )
        return ctx

    #  Node 5: Bid Sizing 
    def _node_bid_sizing(self, ctx: AuctionContext) -> AuctionContext:


            #required_margin = base_margin + regime_adjustment + risk_penalty
            #max_bid = predicted_value × (1 - required_margin)

        if ctx.skip_decision:
            return ctx

        policy = ctx.policy

        # Dynamic required margin
        regime_adj  = policy.regime_margin_adjustment(ctx.market_regime.value)
        risk_adj    = ctx._composite_risk * 0.05

        # Luxury vehicles: lower margin tolerance (higher resale certainty)
        luxury_adj  = -policy.luxury_tolerance if ctx.vehicle_segment == VehicleSegment.LUXURY else 0.0

        # Hot market: demand higher margin (more competition risk)
        hot_adj     = 0.05 if ctx.market_regime == MarketRegime.HOT else \
                      0.10 if ctx.market_regime == MarketRegime.OVERHEATED else 0.0

        required_margin = max(
            0.05,
            policy.base_margin + regime_adj + risk_adj + luxury_adj + hot_adj
        )

        ctx.max_bid = ctx.predicted_value * (1.0 - required_margin)

        # Conviction-based bid increment scale
        # High conviction → larger increments (be more aggressive to win)
        ctx._conviction_scale = 0.5 + ctx.conviction_score * 1.0

        ctx.reasoning_trace.append(
            f"[BidSizing] required_margin={required_margin:.3f} "
            f"max_bid=${ctx.max_bid:,.0f} "
            f"conviction_scale={ctx._conviction_scale:.2f}"
        )
        return ctx

    # Node 6: Final Decision 
    def _node_final_decision(self, ctx: AuctionContext) -> AuctionContext:

        if ctx.skip_decision:
            ctx.reasoning_trace.append(f"[FinalDecision] SKIP — {ctx.skip_reason}")
            return ctx

        # Current bid too close to max — skip
        if ctx.current_highest_bid >= ctx.max_bid * 0.98:
            ctx.skip_decision = True
            ctx.skip_reason   = "Current bid at max_bid limit"
            ctx.reasoning_trace.append(f"[FinalDecision] SKIP — at limit")
            return ctx

        ctx.reasoning_trace.append(
            f"[FinalDecision] BID  "
            f"conviction={ctx.conviction_score:.3f} "
            f"max_bid=${ctx.max_bid:,.0f}"
        )
        return ctx

    def run(self, ctx: AuctionContext) -> AuctionContext:

        ctx = self._node_market_analysis(ctx)
        ctx = self._node_risk_assessment(ctx)
        ctx = self._node_opportunity_cost(ctx)
        ctx = self._node_conviction_scoring(ctx)
        ctx = self._node_bid_sizing(ctx)
        ctx = self._node_final_decision(ctx)
        return ctx



# PHASE 2 — ELITE STRATEGIC BIDDING ENGINE
#  Computed the actual bid amount using sigmoid-decay strategy
# Key ideas: dynamic bid curves, luxury-aware tolerance, adaptive aggression,
#             future-opportunity preservation, probabilistic conviction


class EliteBiddingEngine:


    # Sigmoid constants
    ALPHA    = 0.65    # steepness
    MIDPOINT = 4       # round at which aggression halves
    MIN_INC  = 50.0    # minimum bid step ($)
    MIN_INC_LUXURY = 200.0  # higher minimum step for luxury (avoids looking weak)

    def compute_bid(
        self,
        ctx: AuctionContext,
        current_highest_bid: float,
        round_number: int
    ) -> float:
        """
        Compute bid for this round.

        Returns 0.0 to pass/skip.
        """
        if ctx.skip_decision:
            return 0.0

        max_bid    = ctx.max_bid
        bankroll   = ctx.bankroll
        headroom   = max_bid - current_highest_bid

        if headroom <= 0 or current_highest_bid >= max_bid * 0.99:
            return 0.0

        # Sigmoid decay
        decay     = 1.0 / (1.0 + math.exp(self.ALPHA * (round_number - self.MIDPOINT)))

        # Conviction and policy scaling
        scale     = ctx._conviction_scale * ctx.policy.aggression_scale

        # Strategy mode modifier
        mode_scale = {"conservative": 0.7, "balanced": 1.0, "aggressive": 1.3}.get(
            ctx.policy.strategy_mode, 1.0
        )

        # Minimum increment: luxury gets larger steps
        min_inc   = self.MIN_INC_LUXURY if ctx.vehicle_segment == VehicleSegment.LUXURY else self.MIN_INC

        increment = max(headroom * decay * scale * mode_scale, min_inc)
        bid       = current_highest_bid + increment
        bid       = min(bid, max_bid)
        bid       = min(bid, bankroll)
        bid       = round(bid, 2)

        return bid



# PHASE 5 — TRUE MULTI-AGENT ECOSYSTEM
#  Specialized agents with distinct perspectives collaborate
#  to reach the best collective decision
# Key ideas: role specialization, collaborative deliberation,
#             ecosystem discussion, shared cognitive memory

class SpecializedAgent:

   # Base class for all specialized agents in the ecosystem.

    #Each agent has:
        #• A distinct ROLE with specialized evaluation criteria
       # • A VOTE: (should_bid, conviction, max_bid_suggestion)
        #• A REASONING: textual explanation


    def __init__(self, name: str, memory: EpisodicMemory):
        self.name   = name
        self.memory = memory
        self.wins   = 0
        self.losses = 0

    def evaluate(
        self,
        ctx: AuctionContext
    ) -> Tuple[bool, float, float, str]:

        raise NotImplementedError


class LuxuryAgent(SpecializedAgent):

    #Specializes in luxury vehicle evaluation.



    def __init__(self, memory: EpisodicMemory):
        super().__init__("LuxuryAgent", memory)

    def evaluate(self, ctx: AuctionContext) -> Tuple[bool, float, float, str]:
        if ctx.vehicle_segment not in (VehicleSegment.LUXURY, VehicleSegment.EXOTIC):
            return False, 0.0, 0.0, "Outside luxury domain — abstaining"

        margin = 0.08   # lower margin tolerance for luxury
        max_bid = ctx.predicted_value * (1 - margin)

        # Luxury-specific conviction boost
        seg_win_rate = self.memory.win_rate("luxury")
        conviction   = min(0.95, 0.65 + seg_win_rate * 0.3)

        if ctx.current_highest_bid >= max_bid * 0.95:
            return False, 0.0, max_bid, "Bid too close to luxury max"

        reason = (
            f"Luxury opportunity: pred=${ctx.predicted_value:,.0f} "
            f"seg_win_rate={seg_win_rate:.2f} "
            f"conviction={conviction:.2f} "
            f"max_bid=${max_bid:,.0f}"
        )
        return True, conviction, max_bid, reason


class MarginHunterAgent(SpecializedAgent):
    """Only bids on highest-ROI opportunities."""



    MIN_MARGIN = 0.20

    def __init__(self, memory: EpisodicMemory):
        super().__init__("MarginHunterAgent", memory)

    def evaluate(self, ctx: AuctionContext) -> Tuple[bool, float, float, str]:
        margin = (ctx.predicted_value - ctx.current_highest_bid) / max(ctx.predicted_value, 1)

        if margin < self.MIN_MARGIN:
            return False, 0.0, 0.0, f"Margin {margin:.1%} below 20% threshold"

        max_bid = ctx.predicted_value * (1 - self.MIN_MARGIN)

        # High ROI signal
        if margin > 0.30:
            conviction = 0.90
        elif margin > 0.25:
            conviction = 0.75
        else:
            conviction = 0.60

        return True, conviction, max_bid, (
            f"High ROI opportunity: margin={margin:.1%} conviction={conviction:.2f}"
        )


class RiskAverseAgent(SpecializedAgent):

    #Capital preservation specialist.



    def __init__(self, memory: EpisodicMemory):
        super().__init__("RiskAverseAgent", memory)

    def evaluate(self, ctx: AuctionContext) -> Tuple[bool, float, float, str]:
        bankroll_ratio = ctx.bankroll / max(ctx.starting_bankroll, 1)

        # Hard veto conditions
        if bankroll_ratio < 0.35:
            return False, 0.0, 0.0, f"VETO: Bankroll at {bankroll_ratio:.1%} — capital preservation"

        if ctx.market_regime == MarketRegime.OVERHEATED:
            return False, 0.0, 0.0, "VETO: Overheated market — extreme risk"

        # Conservative margin
        margin = 0.18 if ctx.market_regime == MarketRegime.HOT else 0.12
        max_bid = ctx.predicted_value * (1 - margin)

        if ctx.current_highest_bid >= max_bid * 0.90:
            return False, 0.0, max_bid, "Bid approaching conservative limit"

        conviction = 0.5 * bankroll_ratio  # conviction proportional to bankroll health
        return True, conviction, max_bid, (
            f"Risk-cleared: bankroll={bankroll_ratio:.1%} margin={margin:.1%}"
        )


class MarketObserverAgent(SpecializedAgent):

   # Market intelligence specialist.



    def __init__(self, memory: EpisodicMemory, forecast: FutureMarketIntelligence):
        super().__init__("MarketObserverAgent", memory)
        self.forecast = forecast

    def evaluate(self, ctx: AuctionContext) -> Tuple[bool, float, float, str]:
        hot_prob        = self.forecast.forecast["future_regime_hot_prob"]
        opp_density     = self.forecast.forecast["opportunity_density"]
        price_trend     = self.forecast.forecast["price_trend"]

        # Market timing signal
        if hot_prob > 0.7:
            return False, 0.0, 0.0, (
                f"Market warning: future hot_prob={hot_prob:.2f} — advise waiting"
            )

        if opp_density > 0.6 and price_trend < 0.0:
            # Market cooling + high opportunity density = strong buy signal
            conviction = min(0.9, opp_density)
            max_bid    = ctx.predicted_value * 0.88
            return True, conviction, max_bid, (
                f"Favorable market: opp_density={opp_density:.2f} trend={price_trend:.3f}"
            )

        conviction = 0.5 * (1 - hot_prob) + 0.2
        max_bid    = ctx.predicted_value * 0.87
        return True, conviction, max_bid, (
            f"Neutral market: hot_prob={hot_prob:.2f}"
        )


class OpportunityPreserverAgent(SpecializedAgent):

    #Future opportunity preservation specialist.



    def __init__(self, memory: EpisodicMemory, forecast: FutureMarketIntelligence):
        super().__init__("OpportunityPreserverAgent", memory)
        self.forecast = forecast

    def evaluate(self, ctx: AuctionContext) -> Tuple[bool, float, float, str]:
        future_opp_prob = self.forecast.forecast["future_opportunity_prob"]
        future_score    = self.forecast.forecast["expected_future_profit_score"]
        session_pos     = ctx.session_position

        # In closing stage: nothing to preserve for
        if ctx.auction_stage == AuctionStage.CLOSING or session_pos > 0.85:
            return True, 0.7, ctx.predicted_value * 0.90, \
                "Closing stage — opportunity preservation relaxed"

        # Strong future signal: preserve
        if future_score > 0.7 and future_opp_prob > 0.6:
            current_margin = (ctx.predicted_value - ctx.current_highest_bid) / max(ctx.predicted_value, 1)
            if current_margin < 0.20:
                return False, 0.0, 0.0, (
                    f"PRESERVE: future_score={future_score:.2f} "
                    f"future_opp_prob={future_opp_prob:.2f} > current margin"
                )

        conviction = 0.4 + (1 - future_opp_prob) * 0.4
        return True, conviction, ctx.predicted_value * 0.88, (
            f"Opportunity assessment: future_score={future_score:.2f} session={session_pos:.1%}"
        )


class CommunicationHub:

    #Central hub for inter-agent communication and shared memory.



    def __init__(self):
        self.shared_state: Dict[str, Any] = {
            "market_heat":         0.5,
            "recent_luxury_demand": 0.5,
            "inflation_trend":     0.0,
            "failed_bid_patterns": [],
            "current_regime":      "normal",
            "session_wins":        0,
            "session_profit":      0.0,
            "ecosystem_alert":     None,
        }
        self._messages: deque = deque(maxlen=100)

    def broadcast(self, agent_name: str, message: str, data: Optional[Dict] = None):
        entry = {
            "from":    agent_name,
            "message": message,
            "data":    data or {},
            "ts":      time.time()
        }
        self._messages.append(entry)

    def update_state(self, key: str, value: Any):
        self.shared_state[key] = value

    def get_state(self, key: str, default: Any = None):
        return self.shared_state.get(key, default)

    def recent_messages(self, n: int = 10) -> List[Dict]:
        return list(self._messages)[-n:]


class MultiAgentEcosystem:

   # Phase 5: Orchestrates all specialized agents via collaborative deliberation.
   # Each agent evaluates independently and produces (vote, conviction, max_bid, reason)


    VETO_AGENTS = {"RiskAverseAgent"}   # these can kill a bid entirely

    def __init__(
        self,
        memory: EpisodicMemory,
        forecast: FutureMarketIntelligence
    ):
        self.agents: Dict[str, SpecializedAgent] = {
            "LuxuryAgent":              LuxuryAgent(memory),
            "MarginHunterAgent":        MarginHunterAgent(memory),
            "RiskAverseAgent":          RiskAverseAgent(memory),
            "MarketObserverAgent":      MarketObserverAgent(memory, forecast),
            "OpportunityPreserverAgent":OpportunityPreserverAgent(memory, forecast),
        }
        self.hub = CommunicationHub()

    def deliberate(self, ctx: AuctionContext) -> Tuple[bool, float, float, str]:

        votes: List[Tuple[str, bool, float, float, str]] = []

        for name, agent in self.agents.items():
            try:
                should_bid, conviction, suggested_max, reason = agent.evaluate(ctx)
            except Exception as e:
                should_bid, conviction, suggested_max, reason = False, 0.0, 0.0, f"Error: {e}"

            votes.append((name, should_bid, conviction, suggested_max, reason))
            self.hub.broadcast(name, reason, {"conviction": conviction, "should_bid": should_bid})

        # Hard veto check
        for name, should_bid, conviction, suggested_max, reason in votes:
            if name in self.VETO_AGENTS and not should_bid:
                summary = f"[Ecosystem] VETO by {name}: {reason}"
                return False, 0.0, 0.0, summary

        # Relevance filtering 
        # Agents that abstain (conviction=0) don't influence the vote
        active_votes = [(n, b, c, m, r) for n, b, c, m, r in votes if c > 0]
        if not active_votes:
            return False, 0.0, 0.0, "[Ecosystem] No active votes — skip"

        # Weighted consensus 
        total_weight    = sum(c for _, _, c, _, _ in active_votes)
        bid_weight      = sum(c for _, b, c, _, _ in active_votes if b)
        consensus_ratio = bid_weight / max(total_weight, 0.001)

        # Weighted average of suggested max bids
        if bid_weight > 0:
            recommended_max = sum(m * c for _, b, c, m, _ in active_votes if b and m > 0) / max(bid_weight, 0.001)
        else:
            recommended_max = 0.0

        collective_conviction = bid_weight / max(total_weight, 0.001)
        should_bid = consensus_ratio > 0.50  # majority weighted vote

        vote_summary = " | ".join(
            f"{n[:6]}:{'m' if b else 'n'}({c:.2f})"
            for n, b, c, _, _ in votes
        )
        summary = f"[Ecosystem] {vote_summary} → consensus={consensus_ratio:.2f}"

        return should_bid, collective_conviction, recommended_max, summary



# PHASE 7 — META-STRATEGIC GOVERNANCE
#  Ecosystem-level optimization and dynamic resource allocation
# Key ideas: utility optimization, strategic rotation, dynamic capital
#             allocation, global coordination, adaptive governance

class MetaStrategicGovernor:

   # Phase 7: Top-level coordinator that governs the entire ecosystem.



    def __init__(self, starting_bankroll: float = 500_000):
        self.starting_bankroll  = starting_bankroll
        self.bankroll           = starting_bankroll

        # Capital allocation policy: fraction of bankroll per segment
        self._capital_allocation: Dict[str, float] = {
            "economy":   0.20,
            "mid_range": 0.45,
            "luxury":    0.30,
            "exotic":    0.05,
        }

        # Strategy rotation state
        self._current_strategy = "balanced"
        self._rotation_history: List[str] = ["balanced"]
        self._performance_window: deque = deque(maxlen=30)

        # Session tracking
        self._session_wins     = 0
        self._session_losses   = 0
        self._session_profit   = 0.0
        self._session_cars     = 0

        # Governance alerts
        self._alerts: List[str] = []

    def pre_auction_directive(
        self,
        segment: VehicleSegment,
        forecast: FutureMarketIntelligence,
        memory: EpisodicMemory
    ) -> Dict[str, Any]:

        bankroll_ratio   = self.bankroll / max(self.starting_bankroll, 1)
        segment_budget   = self._capital_allocation.get(segment.value, 0.25)
        allocated_capital = self.bankroll * segment_budget

        # Ecosystem health check
        health = self._compute_ecosystem_health(memory, forecast)

        # Strategy rotation decision
        new_strategy = self._evaluate_strategy_rotation(memory, forecast, bankroll_ratio)
        if new_strategy != self._current_strategy:
            self._current_strategy = new_strategy
            self._rotation_history.append(new_strategy)
            logger.info(f"[Governor] Strategy rotated to: {new_strategy}")

        # Governance override: emergency conservation
        emergency_conservation = bankroll_ratio < 0.25

        directive = {
            "strategy_mode":         self._current_strategy,
            "allocated_capital":     allocated_capital,
            "segment_budget_pct":    segment_budget,
            "ecosystem_health":      health,
            "emergency_conservation":emergency_conservation,
            "max_bid_cap":           allocated_capital * 0.80,  # don't allocate >80% of segment budget
            "session_performance": {
                "wins":    self._session_wins,
                "losses":  self._session_losses,
                "profit":  self._session_profit,
                "win_rate":self._session_wins / max(self._session_wins + self._session_losses, 1),
            }
        }
        return directive

    def post_auction_update(self, record: AuctionRecord):
        """Update governance state after each auction."""
        self._session_cars += 1
        if record.won:
            self._session_wins  += 1
            self._session_profit += record.profit_if_won
            self.bankroll        -= record.actual_price
        else:
            self._session_losses += 1

        self._performance_window.append({
            "won":    record.won,
            "profit": record.profit_if_won if record.won else 0,
            "segment":record.vehicle_segment,
        })

        # Dynamic capital reallocation every 20 auctions
        if self._session_cars % 20 == 0:
            self._rebalance_capital_allocation()

    def _compute_ecosystem_health(
        self,
        memory: EpisodicMemory,
        forecast: FutureMarketIntelligence
    ) -> float:
        """
        0.0 = critical, 1.0 = optimal.
        Combines: win rate, profit rate, bankroll health, market outlook.
        """
        win_rate       = memory.win_rate()
        bankroll_ratio = self.bankroll / max(self.starting_bankroll, 1)
        future_opp     = forecast.forecast["future_opportunity_prob"]

        health = (
            0.35 * win_rate +
            0.35 * bankroll_ratio +
            0.30 * future_opp
        )
        return round(health, 4)

    def _evaluate_strategy_rotation(
        self,
        memory: EpisodicMemory,
        forecast: FutureMarketIntelligence,
        bankroll_ratio: float
    ) -> str:

       # Decide whether to rotate strategy.

        #Rotation logic:
           # • conservative: bankroll < 40% OR recent win_rate < 25%
            #• aggressive:   bankroll > 75% AND win_rate > 65% AND cold market
            #• balanced:     default

        win_rate   = memory.win_rate()
        hot_prob   = forecast.forecast["future_regime_hot_prob"]

        if bankroll_ratio < 0.40 or win_rate < 0.25:
            return "conservative"
        elif bankroll_ratio > 0.75 and win_rate > 0.65 and hot_prob < 0.35:
            return "aggressive"
        else:
            return "balanced"

    def _rebalance_capital_allocation(self):

        #Shift capital allocation toward high-performing segments.

        #Uses recent performance window to identify which segments
        #have generated the best returns, and shifts 5% toward them.

        if not self._performance_window:
            return

        recent = list(self._performance_window)[-20:]
        seg_profit: Dict[str, float] = defaultdict(float)
        seg_count:  Dict[str, int]   = defaultdict(int)

        for p in recent:
            seg_profit[p["segment"]] += p["profit"]
            seg_count[p["segment"]]  += 1

        # Normalize to proportional allocation adjustment
        total_profit = sum(v for v in seg_profit.values() if v > 0) or 1.0
        for seg in self._capital_allocation:
            if seg in seg_profit and seg_profit[seg] > 0:
                self._capital_allocation[seg] = (
                    0.85 * self._capital_allocation[seg] +
                    0.15 * (seg_profit[seg] / total_profit)
                )

        # Renormalize to sum to 1.0
        total = sum(self._capital_allocation.values())
        for seg in self._capital_allocation:
            self._capital_allocation[seg] /= total

        logger.info(f"[Governor] Capital rebalanced: {self._capital_allocation}")



# MASTER BIDDING AGENT — INTEGRATING ALL PHASES
# This is the single class the arena will instantiate.


class BiddingAgent:

    #Master Autonomous Bidding Agent.

    #Integrates Phases 2–7 into a single unified system:
       # Phase 2 → Elite bidding engine (sigmoid-decay curves)
       # Phase 3 → Memory-driven adaptive learning
       # Phase 4 → Strategic reasoning pipeline (LangGraph-style)
       # Phase 5 → Multi-agent ecosystem with collaborative deliberation
       # Phase 6 → LSTM future market intelligence
       # Phase 7 → Meta-strategic governance



    STARTING_BANKROLL = 500_000
    BANKROLL_GUARD    = 0.20   # absolute floor: never go below 20% of starting

    def __init__(self):
        #  Load prediction model & encoder
        _dir         = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()

        model_path = "/kaggle/input/modell/model_VITIKA.cbm"
        encoder_path = "/kaggle/input/encoder/encoders_VITIKA.pkl"

        # Step 1: Load model (pickle)
        if os.path.exists(model_path):
            with open(model_path, "rb") as f:
                self._model = pickle.load(f)
            logger.info("model_vitika.pkl loaded successfully")
        else:
            self._model = None
            logger.warning("model_vitika.pkl not found — using fallback heuristic predictor")

        # Step 1: Load encoders (sklearn pipeline / ColumnTransformer)
        if os.path.exists(encoder_path):
            with open(encoder_path, "rb") as f:
                self._encoders = pickle.load(f)
            logger.info("encoders_vitika.pkl loaded successfully")
        else:
            self._encoders = None
            logger.warning("encoders_vitika.pkl not found — using raw features")

        # Keep _enc as empty dict for backward-compat with helper methods
        self._enc = {}

        #  Initialize all phases 
        self._memory    = EpisodicMemory(max_episodes=500)
        self._policy    = AdaptivePolicyEngine()
        self._forecast  = FutureMarketIntelligence()
        self._reasoner  = StrategicReasoningPipeline(self._memory, self._forecast)
        self._bidder    = EliteBiddingEngine()
        self._ecosystem = MultiAgentEcosystem(self._memory, self._forecast)
        self._governor  = MetaStrategicGovernor(starting_bankroll=self.STARTING_BANKROLL)

        # Session state 
        self.bankroll           = self.STARTING_BANKROLL
        self.predicted_value    = 0.0
        self.vehicle_segment    = VehicleSegment.MID_RANGE
        self.market_regime      = MarketRegime.NORMAL
        self.auction_stage      = AuctionStage.MID
        self.session_position   = 0.0

        # Per-auction state
        self._round_number      = 0
        self._ctx: Optional[AuctionContext] = None
        self._cars_analyzed     = 0
        self._cars_won          = 0
        self._total_profit      = 0.0
        self._total_spent       = 0.0

        logger.info("BiddingAgent initialized — all phases active")


    # INTERNAL: Single-row preprocessing (no df.mean(), no df.groupby())

    def _preprocess(self, car: Dict[str, Any]) -> Any:

        # Step 3: Convert single dict → single-row DataFrame
        car_df = pd.DataFrame([car])

        # Step 4: Apply saved encoders
        if self._encoders is not None:
            try:
                car_encoded = self._encoders.transform(car_df)
                return car_encoded
            except Exception as e:
                logger.error(f"Encoder transform failed: {e}. Falling back to raw df.")
                return car_df
        else:
            # No encoder available — return raw df (model may still handle it)
            return car_df

    def _determine_segment(self, car: Dict[str, Any]) -> VehicleSegment:
        make = str(car.get("make", "")).strip().lower()
        if make in EXOTIC_BRANDS:
            return VehicleSegment.EXOTIC
        elif make in LUXURY_BRANDS:
            return VehicleSegment.LUXURY
        price_hint = self.predicted_value
        if price_hint > 35_000:
            return VehicleSegment.LUXURY
        elif price_hint > 12_000:
            return VehicleSegment.MID_RANGE
        else:
            return VehicleSegment.ECONOMY

    def _update_market_context(self):
        """Infer market regime from recent forecast data."""
        hot_prob = self._forecast.forecast["future_regime_hot_prob"]
        if hot_prob > 0.75:   self.market_regime = MarketRegime.OVERHEATED
        elif hot_prob > 0.55: self.market_regime = MarketRegime.HOT
        elif hot_prob > 0.35: self.market_regime = MarketRegime.WARM
        elif hot_prob > 0.15: self.market_regime = MarketRegime.NORMAL
        else:                 self.market_regime = MarketRegime.COLD

    def _determine_auction_stage(self) -> AuctionStage:
        sp = self.session_position
        if sp < 0.20:   return AuctionStage.OPENING
        elif sp < 0.70: return AuctionStage.MID
        elif sp < 0.90: return AuctionStage.LATE
        else:           return AuctionStage.CLOSING


    # PUBLIC API — Task spec interface

    def analyze_item(self, car: Dict[str, Any]) -> float:

        self._round_number   = 0
        self._cars_analyzed += 1
        self.session_position = min(1.0, self._cars_analyzed / 1000.0)
        self.auction_stage    = self._determine_auction_stage()

        if self._model is not None:
            try:
                # Step 5: Predict price (model inference)
                car_encoded = self._preprocess(car)
                raw_pred = self._model.predict(car_encoded)
                pred_val = float(raw_pred[0]) if hasattr(raw_pred, '__len__') else float(raw_pred)
                # Handle case where model was trained on log1p target
                if pred_val < 50:   # suspiciously small → likely log scale
                    pred_val = float(np.expm1(pred_val))
                self.predicted_value = max(500.0, pred_val)
            except Exception as e:
                logger.error(f"Model prediction failed: {e}")
                self.predicted_value = self._heuristic_price(car)
        else:
            self.predicted_value = self._heuristic_price(car)

        self.vehicle_segment = self._determine_segment(car)
        self._update_market_context()

        # Get governance directive
        directive = self._governor.pre_auction_directive(
            self.vehicle_segment, self._forecast, self._memory
        )
        # Apply strategy mode from governor
        self._policy.strategy_mode = directive["strategy_mode"]

        # Build auction context for this car
        self._ctx = AuctionContext(
            car_features=car,
            predicted_value=self.predicted_value,
            vehicle_segment=self.vehicle_segment,
            bankroll=self.bankroll,
            starting_bankroll=self.STARTING_BANKROLL,
            current_highest_bid=0.0,
            round_number=0,
            market_regime=self.market_regime,
            auction_stage=self.auction_stage,
            session_position=self.session_position,
            memory_records=[],
            policy=self._policy,
            forecast=self._forecast.forecast,
        )

        # Run strategic reasoning pipeline
        self._ctx = self._reasoner.run(self._ctx)

        # Run multi-agent ecosystem deliberation
        eco_bid, eco_conviction, eco_max_bid, eco_summary = \
            self._ecosystem.deliberate(self._ctx)

        # Merge ecosystem recommendation into context
        if not eco_bid and not self._ctx.skip_decision:
            self._ctx.skip_decision = True
            self._ctx.skip_reason   = f"Ecosystem: {eco_summary}"
        elif eco_bid and eco_max_bid > 0:
            # Take the more conservative of reasoner and ecosystem max bids
            self._ctx.max_bid = min(
                self._ctx.max_bid if self._ctx.max_bid > 0 else eco_max_bid,
                eco_max_bid
            )
            self._ctx.conviction_score = (
                0.5 * self._ctx.conviction_score + 0.5 * eco_conviction
            )

        logger.debug(f"analyze_item: pred=${self.predicted_value:,.0f} "
                     f"segment={self.vehicle_segment.value} "
                     f"skip={self._ctx.skip_decision}")
        return self.predicted_value

    def place_bid(self, current_highest_bid: float) -> float:
        """
        Deterministic bid placement.

        Pure function of:
            self.predicted_value
            self.bankroll
            current_highest_bid
            self._round_number  (tracked internally)

        Returns 0.0 to pass.
        """
        self._round_number += 1

        if self._ctx is None:
            return 0.0

        # Update context with current bid state
        self._ctx.current_highest_bid = current_highest_bid
        self._ctx.round_number        = self._round_number

        # Hard guards (non-negotiable)
        bankroll_floor = self.STARTING_BANKROLL * self.BANKROLL_GUARD
        if self.bankroll <= bankroll_floor:
            return 0.0
        if self.bankroll < current_highest_bid:
            return 0.0
        if self._ctx.skip_decision:
            return 0.0

        # Governor: apply segment budget cap
        directive  = self._governor.pre_auction_directive(
            self.vehicle_segment, self._forecast, self._memory
        )
        if current_highest_bid > directive["max_bid_cap"]:
            return 0.0

        # Compute bid via elite bidding engine
        bid = self._bidder.compute_bid(self._ctx, current_highest_bid, self._round_number)
        return bid

    def record_win(self, hammer_price: float, resale_value: Optional[float] = None):

        resale         = resale_value or self.predicted_value
        profit         = resale - hammer_price
        self.bankroll -= hammer_price
        self._total_spent  += hammer_price
        self._total_profit += profit
        self._cars_won     += 1

        # Build record for memory + learning
        competition_proxy = min(1.0, hammer_price / max(self.predicted_value * 0.7, 1) - 1)
        record = AuctionRecord(
            car_id=f"car_{self._cars_analyzed}",
            features=self._ctx.car_features if self._ctx else {},
            predicted_value=self.predicted_value,
            actual_price=hammer_price,
            our_final_bid=hammer_price,
            won=True,
            profit_if_won=profit,
            margin_pct=profit / max(resale, 1),
            market_regime=self.market_regime.value,
            auction_stage=self.auction_stage.value,
            vehicle_segment=self.vehicle_segment.value,
            rounds_taken=self._round_number,
            competition_proxy=max(0.0, competition_proxy),
        )
        record._bankroll_ratio = self.bankroll / self.STARTING_BANKROLL

        self._memory.store(record)
        self._policy.update(record, self._memory)
        self._governor.post_auction_update(record)
        self._forecast.update(
            hammer_price, competition_proxy, self.predicted_value / 50000,
            True, self.market_regime.value
        )

        logger.info(
            f"[WIN #{self._cars_won}] "
            f"Paid: ${hammer_price:,.0f}  "
            f"Est.Value: ${resale:,.0f}  "
            f"Profit: ${profit:,.0f}  "
            f"Bankroll: ${self.bankroll:,.0f}"
        )

    def record_loss(self, hammer_price: float):

        profit_missed  = self.predicted_value - hammer_price
        regret         = max(0.0, profit_missed)
        competition_proxy = min(1.0, hammer_price / max(self.predicted_value * 0.7, 1) - 1)

        record = AuctionRecord(
            car_id=f"car_{self._cars_analyzed}_loss",
            features=self._ctx.car_features if self._ctx else {},
            predicted_value=self.predicted_value,
            actual_price=hammer_price,
            our_final_bid=self._ctx.max_bid if self._ctx else 0.0,
            won=False,
            profit_if_won=max(0.0, profit_missed),
            margin_pct=-(hammer_price - self.predicted_value) / max(self.predicted_value, 1),
            market_regime=self.market_regime.value,
            auction_stage=self.auction_stage.value,
            vehicle_segment=self.vehicle_segment.value,
            rounds_taken=self._round_number,
            competition_proxy=max(0.0, competition_proxy),
            regret=regret,
        )
        record._bankroll_ratio = self.bankroll / self.STARTING_BANKROLL

        self._memory.store(record)
        self._policy.update(record, self._memory)
        self._governor.post_auction_update(record)
        self._forecast.update(
            hammer_price, competition_proxy, self.predicted_value / 50000,
            False, self.market_regime.value
        )

    def _heuristic_price(self, car: Dict[str, Any]) -> float:

        make  = str(car.get("make", "")).lower()
        year  = int(float(car.get("year", 2015)))
        odo   = float(car.get("odometer", 50_000))
        cond  = float(car.get("condition", 3.0))

        base  = 8_000
        base += max(0, (year - 2010)) * 1_200   # newer = more valuable
        base -= (odo / 1_000) * 50               # mileage discount
        base += (cond - 2.5) * 2_000             # condition premium
        base *= 1.5 if make in LUXURY_BRANDS else 1.0
        return max(1_000.0, float(base))

    def status(self) -> Dict[str, Any]:

        roi = (self._total_profit / max(self._total_spent, 1)) * 100
        win_rate = self._cars_won / max(self._cars_analyzed, 1)

        status = {
            "cars_analyzed":    self._cars_analyzed,
            "cars_won":         self._cars_won,
            "win_rate_pct":     round(win_rate * 100, 2),
            "total_spent":      round(self._total_spent, 2),
            "total_profit_est": round(self._total_profit, 2),
            "roi_pct":          round(roi, 2),
            "bankroll":         round(self.bankroll, 2),
            "strategy_mode":    self._policy.strategy_mode,
            "market_regime":    self.market_regime.value,
            "auction_stage":    self.auction_stage.value,
            "ecosystem_health": self._governor._compute_ecosystem_health(self._memory, self._forecast),
            "memory_episodes":  len(self._memory.episodes),
            "lstm_trained":     self._forecast._model_trained,
            "capital_alloc":    self._governor._capital_allocation,
        }

        print("\n" + "=" * 60)
        print("  AUTONOMOUS BIDDING ECOSYSTEM — STATUS REPORT")
        print("=" * 60)
        for k, v in status.items():
            print(f"  {k:25s}: {v}")
        print("=" * 60)
        return status



# DEMONSTRATION / QUICK Test 

def run_simulation_demo():

   # Simulate 20 auction rounds to verify the full pipeline works end-to-end.
   # Does not require the trained model file — uses heuristic fallback.

    print("\n" + "=" * 70)
    print("  AUTONOMOUS AUCTION ECOSYSTEM — agent_vitika.py — DEMO (Phase 2–7)")
    print("=" * 70)

    agent = BiddingAgent()

    test_cars = [
        {"year": 2019, "make": "Toyota",        "model": "Camry",     "trim": "SE",     "body": "Sedan",   "transmission": "automatic", "state": "ca", "condition": 3.5, "odometer": 42_000, "color": "white",   "interior": "black"},
        {"year": 2020, "make": "BMW",            "model": "3 Series",  "trim": "Sport",  "body": "Sedan",   "transmission": "automatic", "state": "ny", "condition": 4.2, "odometer": 28_000, "color": "black",   "interior": "black"},
        {"year": 2017, "make": "Ford",           "model": "F-150",     "trim": "XLT",    "body": "Pickup",  "transmission": "automatic", "state": "tx", "condition": 3.8, "odometer": 65_000, "color": "silver",  "interior": "gray"},
        {"year": 2015, "make": "Honda",          "model": "Accord",    "trim": "EX",     "body": "Sedan",   "transmission": "automatic", "state": "fl", "condition": 2.8, "odometer": 88_000, "color": "blue",    "interior": "beige"},
        {"year": 2021, "make": "Lexus",          "model": "RX",        "trim": "350",    "body": "SUV",     "transmission": "automatic", "state": "ca", "condition": 4.8, "odometer": 15_000, "color": "white",   "interior": "tan"},
        {"year": 2018, "make": "Chevrolet",      "model": "Silverado", "trim": "LT",     "body": "Pickup",  "transmission": "automatic", "state": "tx", "condition": 3.2, "odometer": 55_000, "color": "gray",    "interior": "gray"},
        {"year": 2022, "make": "Audi",           "model": "Q5",        "trim": "Premium","body": "SUV",     "transmission": "automatic", "state": "wa", "condition": 4.5, "odometer": 18_000, "color": "black",   "interior": "black"},
        {"year": 2016, "make": "Nissan",         "model": "Altima",    "trim": "S",      "body": "Sedan",   "transmission": "automatic", "state": "ga", "condition": 2.5, "odometer": 95_000, "color": "silver",  "interior": "beige"},
        {"year": 2020, "make": "Jeep",           "model": "Wrangler",  "trim": "Sport",  "body": "SUV",     "transmission": "manual",    "state": "co", "condition": 4.0, "odometer": 32_000, "color": "red",     "interior": "black"},
        {"year": 2019, "make": "mercedes-benz",  "model": "C-Class",   "trim": "C300",   "body": "Sedan",   "transmission": "automatic", "state": "fl", "condition": 4.3, "odometer": 35_000, "color": "silver",  "interior": "black"},
    ] * 2   # 20 cars total

    print(f"\n{'Car':40s}  {'Predicted':>12}  {'Bid R1':>10}  {'Bid R5':>10}  {'Skip?':>8}")
    print("-" * 90)

    wins = 0
    for i, car in enumerate(test_cars):
        pred   = agent.analyze_item(car)
        bid_r1 = agent.place_bid(pred * 0.50)    # simulate opening bid
        bid_r5 = agent.place_bid(pred * 0.75)    # simulate mid auction

        # Simulate win/loss (win if we bid > 80% of predicted)
        hammer = pred * 0.82
        label  = f"{car['year']} {car['make'].title()} {car['model'].title()}"

        skip = "SKIP" if bid_r1 == 0.0 else ""
        print(f"{label:40s}  ${pred:>11,.0f}  ${bid_r1:>9,.0f}  ${bid_r5:>9,.0f}  {skip:>8}")

        if bid_r5 > 0 and bid_r5 >= hammer:
            agent.record_win(hammer)
            wins += 1
        else:
            agent.record_loss(hammer)

    print(f"\n{'='*90}")
    print(f"Simulation complete: {wins}/{len(test_cars)} wins")
    agent.status()


if __name__ == "__main__":
    run_simulation_demo()


2026-05-28 22:46:37,979 [WARNING] AuctionEcosystem — model_vitika.pkl not found — using fallback heuristic predictor
2026-05-28 22:46:37,980 [WARNING] AuctionEcosystem — encoders_vitika.pkl not found — using raw features
2026-05-28 22:46:37,981 [INFO] AuctionEcosystem — BiddingAgent initialized — all phases active
2026-05-28 22:46:37,983 [INFO] AuctionEcosystem — [WIN #1] Paid: $15,334  Est.Value: $18,700  Profit: $3,366  Bankroll: $484,666
2026-05-28 22:46:37,984 [INFO] AuctionEcosystem — [Governor] Strategy rotated to: aggressive
2026-05-28 22:46:37,985 [INFO] AuctionEcosystem — [WIN #2] Paid: $27,060  Est.Value: $33,000  Profit: $5,940  Bankroll: $457,606
2026-05-28 22:46:37,986 [INFO] AuctionEcosystem — [WIN #3] Paid: $12,915  Est.Value: $15,750  Profit: $2,835  Bankroll: $444,691
2026-05-28 22:46:37,987 [INFO] AuctionEcosystem — [WIN #4] Paid: $8,364  Est.Value: $10,200  Profit: $1,836  Bankroll: $436,327
2026-05-28 22:46:37,988 [INFO] AuctionEcosystem — [WIN #5] Paid: $30,811  Es


  AUTONOMOUS AUCTION ECOSYSTEM — agent_vitika.py — DEMO (Phase 2–7)

Car                                          Predicted      Bid R1      Bid R5     Skip?
------------------------------------------------------------------------------------------
2019 Toyota Camry                         $     18,700  $   15,413  $   15,583          
2020 Bmw 3 Series                         $     33,000  $   28,596  $   28,596          
2017 Ford F-150                           $     15,750  $   13,384  $   13,384          
2015 Honda Accord                         $     10,200  $    8,644  $    8,644          
2021 Lexus Rx                             $     37,575  $   32,619  $   32,619          
2018 Chevrolet Silverado                  $     16,250  $   13,765  $   13,765          
2022 Audi Q5                              $     38,250  $   33,194  $   33,194          
2016 Nissan Altima                        $     10,450  $    8,845  $    8,833          
2020 Jeep Wrangler                    

2026-05-28 22:46:41,623 [INFO] AuctionEcosystem — LSTM market intelligence model updated.
2026-05-28 22:46:41,625 [INFO] AuctionEcosystem — [WIN #16] Paid: $13,325  Est.Value: $16,250  Profit: $2,925  Bankroll: $201,500
2026-05-28 22:46:41,707 [INFO] AuctionEcosystem — LSTM market intelligence model updated.
2026-05-28 22:46:41,709 [INFO] AuctionEcosystem — [WIN #17] Paid: $31,365  Est.Value: $38,250  Profit: $6,885  Bankroll: $170,134
2026-05-28 22:46:41,710 [INFO] AuctionEcosystem — [Governor] Strategy rotated to: conservative
2026-05-28 22:46:41,780 [INFO] AuctionEcosystem — LSTM market intelligence model updated.


2022 Audi Q5                              $     38,250  $   33,267  $   33,267          
2016 Nissan Altima                        $     10,450  $        0  $        0      SKIP
2020 Jeep Wrangler                        $     21,400  $        0  $        0      SKIP


2026-05-28 22:46:41,854 [INFO] AuctionEcosystem — LSTM market intelligence model updated.
2026-05-28 22:46:41,856 [INFO] AuctionEcosystem — [Governor] Capital rebalanced: {'economy': 0.1801521834639927, 'mid_range': 0.4251013119705496, 'luxury': 0.34511871300218216, 'exotic': 0.04962779156327543}
2026-05-28 22:46:41,930 [INFO] AuctionEcosystem — LSTM market intelligence model updated.


2019 Mercedes-Benz C-Class                $     30,975  $        0  $        0      SKIP

Simulation complete: 17/20 wins

  AUTONOMOUS BIDDING ECOSYSTEM — STATUS REPORT
  cars_analyzed            : 20
  cars_won                 : 17
  win_rate_pct             : 85.0
  total_spent              : 329865.5
  total_profit_est         : 72409.5
  roi_pct                  : 21.95
  bankroll                 : 170134.5
  strategy_mode            : conservative
  market_regime            : cold
  auction_stage            : opening
  ecosystem_health         : 0.6743
  memory_episodes          : 20
  lstm_trained             : True
  capital_alloc            : {'economy': 0.1801521834639927, 'mid_range': 0.4251013119705496, 'luxury': 0.34511871300218216, 'exotic': 0.04962779156327543}
